# Ollama Unstructured-Output Evaluation

Evaluate a selected local Ollama model on Calculus I multiple-choice questions. Ollama produces a free-form solution, and `gpt-4o-mini` converts that solution into a structured answer choice for evaluation.

## 1. Imports and configuration

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from ollama import chat
from openai import OpenAI
from pydantic import BaseModel


### Run configuration

Change `MODEL_NAME` to test another Ollama model. The filesystem-safe artifact name, output directory, and every generated filename update automatically. `EXTRACTION_MODEL_NAME` controls the OpenAI model used only for answer-letter extraction.

In [ ]:
# Support launching Jupyter from either the repository root or notebooks/.
WORKING_DIRECTORY = Path.cwd()
PROJECT_ROOT = (
    WORKING_DIRECTORY.parent
    if WORKING_DIRECTORY.name == "notebooks"
    else WORKING_DIRECTORY
)

MODEL_NAME = "llama3.2:1b"
EXTRACTION_MODEL_NAME = "gpt-4o-mini"
MODEL_ARTIFACT_NAME = MODEL_NAME.replace(":", "-").replace("/", "-")
RUN_NAME = f"{MODEL_ARTIFACT_NAME}-subsample"

DATA_PATH = PROJECT_ROOT / "data" / "test3_input_subset_seed42_numq1 (clean testing dataset).csv"
MODEL_OUTPUT_DIRECTORY = PROJECT_ROOT / "output" / MODEL_ARTIFACT_NAME
CHECKPOINT_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-checkpoint.json"
RESULTS_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output.csv"
INCORRECT_RESULTS_PATH = (
    MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output-incorrect.csv"
)

MODEL_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)


## 2. OpenAI extraction client

In [ ]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


## 3. Load and inspect the dataset

In [ ]:
def load_dataset(data_path: Path) -> pd.DataFrame:
    """Load the evaluation dataset."""
    return pd.read_csv(data_path)


def summarize_dataset(dataframe: pd.DataFrame) -> pd.Series:
    """Summarize row counts and duplicate checks used in the original notebook."""
    return pd.Series(
        {
            "rows": len(dataframe),
            "columns": len(dataframe.columns),
            "duplicate_rows": int(dataframe.duplicated().sum()),
            "duplicate_questions": int(dataframe["Question"].duplicated().sum()),
            "duplicate_ids": int(dataframe["ID"].duplicated().sum()),
        }
    )


df = load_dataset(DATA_PATH)
print(f"Loaded {len(df)} questions from {DATA_PATH}")
summarize_dataset(df)

In [ ]:
# Inspect any duplicated rows, questions, or IDs found by the summary above.
duplicate_rows = df.loc[df.duplicated(keep=False)].sort_values(df.columns.tolist())
duplicate_questions = df.loc[df["Question"].duplicated(keep=False)].sort_values(
    "Question"
)
duplicate_ids = df.loc[df["ID"].duplicated(keep=False)].sort_values("ID")

df.describe(include="all")

## 4. Unstructured Ollama response pipeline

### Few-shot example

In [ ]:
EXAMPLE_RESPONSE = '''
"Complete the square to write the equation in the standard form f(x)=a(x-h)^2+k. f(x)=2x^2+12x+25
                a) f(x)=2(x+3)^2+16
                b) f(x)=2(x-6)^2+7
                c) f(x)=2(x+3)^2+7
                d) f(x)=2(x-3)^2+16
                e) f(x)=2(x+3)^2-7
                f) None of the above."
    "detailed_solution_explanation": "To complete the square for f(x) = 2x² + 12x + 25, first factor out 2 from the terms involving x:\n\nf(x) = 2(x² + 6x) + 25.\n\nNext, complete the square inside the parentheses:\n\nx² + 6x = x² + 6x + 9 − 9 = (x + 3)² − 9.\n\nRewrite f(x):\n\nf(x) = 2[(x + 3)² − 9] + 25\n      = 2(x + 3)² − 18 + 25\n      = 2(x + 3)² + 7.\n",
    "correct_option_choice_letter": "c"
'''

SYSTEM_PROMPT = (
    "Act as a mathematician specializing in Calculus. Solve the college-level "
    "Calculus I multiple-choice problem step by step and state the correct "
    "option letter clearly. Respond in plain text."
)


In [ ]:
def load_checkpoint(checkpoint_path: Path) -> dict:
    """Load saved Ollama responses, or return an empty checkpoint."""
    if not checkpoint_path.exists():
        return {}
    with checkpoint_path.open("r", encoding="utf-8") as checkpoint_file:
        return json.load(checkpoint_file)


def save_checkpoint(checkpoint_path: Path, data: dict) -> None:
    """Persist Ollama responses as formatted JSON."""
    with checkpoint_path.open("w", encoding="utf-8") as checkpoint_file:
        json.dump(data, checkpoint_file, indent=4)


In [ ]:
def query_ollama(question: str, model: str = MODEL_NAME) -> str:
    """Request one free-form calculus solution from the local Ollama model."""
    response = chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Use this example as a reference:\n{EXAMPLE_RESPONSE}\n\n"
                    f"Now solve the following question:\n{question}"
                ),
            },
        ],
        options={"temperature": 0},
    )
    return response.message.content or ""


In [ ]:
def collect_responses(
    dataframe: pd.DataFrame,
    checkpoint_path: Path,
    model: str = MODEL_NAME,
) -> dict:
    """Query unanswered rows and checkpoint each free-form Ollama response."""
    checkpoint = load_checkpoint(checkpoint_path)

    for _, row in dataframe.iterrows():
        question_id = str(row["ID"])
        if question_id in checkpoint:
            print(f"Skipping ID {question_id}, already processed.")
            continue

        print(f"Processing ID {question_id}...")
        checkpoint[question_id] = {
            "Long Answer": query_ollama(row["Question"], model=model)
        }
        save_checkpoint(checkpoint_path, checkpoint)

    save_checkpoint(checkpoint_path, checkpoint)
    print(f"Checkpoint saved to {checkpoint_path}")
    return checkpoint


def get_checkpoint_explanation(record: dict) -> str:
    """Read new free-form records and legacy structured Ollama records."""
    return record.get(
        "Long Answer",
        record.get("detailed_solution_explanation", ""),
    )


def add_explanations(dataframe: pd.DataFrame, checkpoint: dict) -> pd.DataFrame:
    """Add checkpointed Ollama solutions to the dataset."""
    result = dataframe.copy()
    result["LLM Answer Explanation"] = result["ID"].astype(str).map(
        lambda question_id: get_checkpoint_explanation(
            checkpoint.get(question_id, {})
        )
    )
    return result


### Run the primary-model pipeline

> **Note:** This cell sends requests to the local Ollama server only for questions absent from the checkpoint. Make sure the selected Ollama model has been pulled before running it.

In [ ]:
df = df.head(10)  # Limit to first 10 rows for testing

checkpoint = collect_responses(df, CHECKPOINT_PATH)
df = add_explanations(df, checkpoint)
df.head(10)


## 5. Extract structured answer choices with OpenAI

Ollama remains unstructured in the primary stage. A separate `gpt-4o-mini` call extracts only the final answer-choice letter from each saved solution.

In [ ]:
class AnswerChoice(BaseModel):
    """Structured answer extracted from a free-form Ollama solution."""

    correct_option_choice_letter: str


def get_correct_option(
    response_text: str, model: str = EXTRACTION_MODEL_NAME
) -> str:
    """Extract an option letter and return the raw structured JSON text."""
    prompt = (
        "Read the following raw response from an LLM and extract the correct "
        f"option letter:\n\n{response_text}\n\n"
        "If you cannot find the correct option letter, output 'NA'."
    )
    parse_completion = getattr(client.chat.completions, "parse", None)
    if parse_completion is None:
        parse_completion = client.beta.chat.completions.parse

    completion = parse_completion(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert annotator. You convert raw text to "
                    "structured text."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        response_format=AnswerChoice,
    )
    return completion.choices[0].message.content or ""


def extract_answer_choices(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Extract a structured answer choice from each Ollama solution."""
    result = dataframe.copy()
    for index, row in result.iterrows():
        explanation = row["LLM Answer Explanation"]
        if pd.isna(explanation) or not explanation:
            continue
        try:
            result.at[index, "LLM Answer Choice (RAW)"] = get_correct_option(
                explanation
            )
        except Exception as error:
            print(f"Error processing ID {row['ID']}: {error}")
            result.at[index, "LLM Answer Choice (RAW)"] = None
    return result


### Run answer extraction

> **Note:** This cell makes one billable `gpt-4o-mini` API call per non-empty Ollama solution.

In [ ]:
df = extract_answer_choices(df)
df.head()


## 6. Normalize and evaluate answers

In [ ]:
def extract_letter_choice(raw_response: str) -> str:
    """Read the option letter from the extraction model's JSON response."""
    if not isinstance(raw_response, str) or not raw_response:
        return ""
    try:
        return json.loads(raw_response).get("correct_option_choice_letter", "")
    except json.JSONDecodeError:
        match = re.search(
            r'"correct_option_choice_letter"\s*:\s*"(.*?)"',
            raw_response,
        )
        return match.group(1) if match else ""


def evaluate_answers(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, float]:
    """Normalize extracted choices and calculate accuracy."""
    result = dataframe.copy()
    result["LLM Answer Choice"] = result["LLM Answer Choice (RAW)"].map(
        extract_letter_choice
    )
    result["Correct"] = (
        result["Correct Answer"] == result["LLM Answer Choice"]
    ).astype(int)
    return result, result["Correct"].mean()

In [ ]:
df, accuracy = evaluate_answers(df)
print(f"Accuracy: {accuracy * 100:.2f}%")
df.head()

## 7. Export results

In [ ]:
incorrect_df = df.loc[df["Correct"] == 0].copy()

incorrect_df.to_csv(INCORRECT_RESULTS_PATH, index=False)
df.to_csv(RESULTS_PATH, index=False)

print(f"Saved {len(df)} results to {RESULTS_PATH}")
print(f"Saved {len(incorrect_df)} incorrect results to {INCORRECT_RESULTS_PATH}")